<a href="https://colab.research.google.com/github/infernoWolf99/spark-rebuild/blob/main/MedNextWithAttention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install torchio monai fvcore
import os, sys
print('Done')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.9/187.9 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 43.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.8 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.3 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 31.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.

In [2]:
# !git clone https://github.com/MIC-DKFZ/MedNeXt.git mednext
# %cd mednext
# !pip -q install -e .
# %cd ..

# sys.path.append('mednext')

In [3]:
!gdown 1BRqrwdm7HE7c07Ky4izoNYgiD9VPvxLB
!gdown 1TyZVcBoA0UWCbXbH-wSzDJs43uh4N__F

Downloading...
From (original): https://drive.google.com/uc?id=1BRqrwdm7HE7c07Ky4izoNYgiD9VPvxLB
From (redirected): https://drive.google.com/uc?id=1BRqrwdm7HE7c07Ky4izoNYgiD9VPvxLB&confirm=t&uuid=09624e55-25a7-41ed-8e0f-85a34948e359
To: /kaggle/working/ASNR-MICCAI-BraTS2023-SSA-Challenge-TrainingData.zip
100%|█████████████████████████████████████████| 741M/741M [00:06<00:00, 118MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1TyZVcBoA0UWCbXbH-wSzDJs43uh4N__F
From (redirected): https://drive.google.com/uc?id=1TyZVcBoA0UWCbXbH-wSzDJs43uh4N__F&confirm=t&uuid=2afb0730-0177-4b96-a611-c6d14b4e0a95
To: /kaggle/working/ASNR-MICCAI-BraTS2023-SSA-Challenge-ValidationData.zip
100%|█████████████████████████████████████████| 149M/149M [00:00<00:00, 169MB/s]


In [10]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch, sys, os
import torch.nn as nn
from fvcore.nn import FlopCountAnalysis
# from fvcore.nn import parameter_count_table
import torchio as tio
from torchio.data import SubjectsLoader, SubjectsDataset
import matplotlib.pyplot as plt
from glob import glob
import numpy as np
from tqdm.notebook import tqdm
from monai.metrics import DiceMetric, HausdorffDistanceMetric
from monai.losses import DiceLoss
from matplotlib.colors import ListedColormap
import warnings
import pandas as pd
from datetime import datetime
import nibabel as nib
import shutil
from monai.metrics import compute_hausdorff_distance

# from pydrive.auth import GoogleAuth
# from pydrive.drive import GoogleDrive
# from google.colab import auth
# from oauth2client.client import GoogleCredentials

# auth.authenticate_user()
# gauth = GoogleAuth()
# gauth.credentials = GoogleCredentials.get_application_default()
# drive = GoogleDrive(gauth)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from IPython.display import FileLink
warnings.filterwarnings("ignore")
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import warnings
warnings.filterwarnings('ignore')

In [11]:
# !mkdir "kaggle/working/training" "kaggle/working/validation"
!unzip -qq /kaggle/working/ASNR-MICCAI-BraTS2023-SSA-Challenge-TrainingData.zip -d training
!unzip -qq /kaggle/working/ASNR-MICCAI-BraTS2023-SSA-Challenge-ValidationData.zip -d validation

replace training/ASNR-MICCAI-BraTS2023-SSA-Challenge-TrainingData_V2/BraTS-SSA-00121-000/BraTS-SSA-00121-000-seg.nii.gz? [y]es, [n]o, [A]ll, [N]one, [r]ename: ^C
replace validation/ASNR-MICCAI-BraTS2023-SSA-Challenge-ValidationData/BraTS-SSA-00188-000/BraTS-SSA-00188-000-t2f.nii.gz? [y]es, [n]o, [A]ll, [N]one, [r]ename: ^C


In [6]:
training_path = '/kaggle/working/training/ASNR-MICCAI-BraTS2023-SSA-Challenge-TrainingData_V2'
validation_path = '/kaggle/working/validation/ASNR-MICCAI-BraTS2023-SSA-Challenge-ValidationData'

In [7]:
print(len(os.listdir(training_path)))
print(len(os.listdir(validation_path)))

60
15


In [8]:
# --- dictinary containing all tunable parameters --- #

def get_out_dir():
    now = datetime.now().strftime("%Y-%m-%d_%H-%M")
    full_path = f'./output/{now}/'
    os.makedirs(full_path, exist_ok=True)
    return full_path

parameters = {
    'kernel_size': 5,        # could also be 2, 5
    'group_size': 4,         # group size of normalization layers...can try using values like 2, 4, 6, 8
    'norm_type': 'group',    # type of normalization used...other supported types are 'instance' and 'layer'
    'activation_fn': 'leaky', # other supported types are 'leaky' and 'swish'
    'lr': 1e-3,              # learning rate
    'optimizer': 'adamw',     # options are 'adam' and 'adamw'
    'num_epochs': 20,
    'scheduler': 'reduceLROnPlateau', # options are 'reduceLROnPlateau' and 'cosineAnnealing'
    'patience': 5,                    # scheduler patience works only when using reduceLRONPlataeu
    'stopping_patience': 20,           # controls early stopping patience
    'out_dir': get_out_dir(),                  # lets use 'Ike' and 'Ahmed'
}
# now = datetime.now()
# !mkdir parameters['out_dir']

In [9]:
class BratsDataset(SubjectsDataset):
    def __init__(self, root_dir: str, transform=None, test=False):
        self.patient_dirs = os.listdir(root_dir)
        self.transform = transform
        self.subjects = []

        for patient_dir in self.patient_dirs:
            t1c_path = glob(os.path.join(root_dir, patient_dir, "*t1c.nii.gz*"), recursive=True)
            t1n_path = glob(os.path.join(root_dir, patient_dir, "*t1n.nii.gz*"), recursive=True)
            t2f_path = glob(os.path.join(root_dir, patient_dir, "*t2f.nii.gz*"), recursive=True)
            t2w_path = glob(os.path.join(root_dir, patient_dir, "*t2w.nii.gz*"), recursive=True)
            seg_path = glob(os.path.join(root_dir, patient_dir, "*seg.nii.gz*"), recursive=True)

            if test and not (t1c_path and t1n_path and t2f_path and t2w_path):
                print(f"Skipping {patient_dir} (missing inputs)")
                continue
            elif not test and not (t1c_path and t1n_path and t2f_path and t2w_path and seg_path):
                print(f"Skipping {patient_dir} (missing inputs or labels)")
                continue

            subject_dict = {
                'vol': tio.ScalarImage(
                    [t1c_path[0], 
                     # t1n_path[0], 
                     t2f_path[0], 
                     # t2w_path[0]
                    ]),
                'patient_id': patient_dir
            }

            if not test:
                subject_dict['label'] = tio.LabelMap(seg_path[0])

            self.subjects.append(tio.Subject(**subject_dict))

        super().__init__(self.subjects, transform=transform)

        
train_transform = tio.Compose([
    tio.RescaleIntensity(out_min_max=(0, 99.5), exclude=["label"]), 
    # tio.Resample(1.0),  
    tio.Resize((128, 128, 64)), 
    tio.ZNormalization(exclude=['label']),
])

test_transform = tio.Compose([
    tio.RescaleIntensity(out_min_max=(0, 99.5)), 
    # tio.Resample(1.0),  
    tio.Resize((128, 128, 64)), 
    tio.ZNormalization(),
])

training_dataset = BratsDataset(root_dir=training_path, transform=train_transform)
test_dataset = BratsDataset(root_dir=validation_path, transform=test_transform, test=True)

train_set, val_set = torch.utils.data.random_split(training_dataset, [.8, .2])

train_loader = SubjectsLoader(train_set, batch_size=2, shuffle=True, num_workers=2)
val_loader = SubjectsLoader(val_set, batch_size=2, shuffle=False, num_workers=2)

test_loader = SubjectsLoader(test_dataset, batch_size=2, shuffle=False, num_workers=2)


In [ ]:
# --- Squeeze-and-Excitation Block (3D) ---
class SEBlock3D(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool3d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _, _ = x.size()
        y = self.pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1, 1)
        return x * y


# --- Axial Attention (3D) ---
class AxialAttention3D(nn.Module):
    def __init__(self, dim, heads=4):
        super().__init__()
        self.heads = heads
        self.scale = (dim // heads) ** -0.5
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x):
        b, c, d, h, w = x.shape
        x = x.permute(0, 2, 3, 4, 1).reshape(b * d * h, w, c)
        qkv = self.qkv(x).chunk(3, dim=-1)
        q, k, v = [t.reshape(t.shape[0], t.shape[1], self.heads, c // self.heads).transpose(1, 2) for t in qkv]
        attn = (q @ k.transpose(-1, -2)) * self.scale
        attn = attn.softmax(dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(x.shape[0], x.shape[1], c)
        out = self.proj(out).reshape(b, d, h, w, c).permute(0, 4, 1, 2, 3)
        return out


# --- MedNeXt Block ---
class MedNeXtBlock(nn.Module):
    def __init__(self, channels, kernel_size=3, act_fn="gelu", norm_type="group", num_groups=8):
        super().__init__()
        self.dw_conv = nn.Conv3d(channels, channels, kernel_size, padding=kernel_size // 2, groups=channels)

        if norm_type == 'instance':
            self.norm = nn.InstanceNorm3d(channels)
        elif norm_type == 'layer':
            self.norm = nn.GroupNorm(1, channels)
        else:
            assert channels % num_groups == 0, "channels must be divisible by num_groups"
            self.norm = nn.GroupNorm(num_groups, channels)

        if act_fn == 'gelu':
            self.act_fn = nn.GELU()
        elif act_fn == 'leaky':
            self.act_fn = nn.LeakyReLU()
        elif act_fn == 'swish':
            self.act_fn = nn.SiLU()
        else:
            raise ValueError('activation function must be "gelu", "leaky", or "swish"')

        self.pw_conv = nn.Conv3d(channels, channels, kernel_size=1)
        self.se = SEBlock3D(channels)

    def forward(self, x):
        out = self.dw_conv(x)
        out = self.norm(out)
        out = self.act_fn(out)
        out = self.pw_conv(out)
        out = self.se(out)
        return out + x


# --- Full Deep MedNeXt U-Net ---
class MedNeXtWithAttentionUNet(nn.Module):
    def __init__(self, in_channels=4, out_channels=4, base_channels=32, use_attention=True):
        super().__init__()
        self.use_attention = use_attention

        self.stem = nn.Conv3d(in_channels, base_channels, kernel_size=3, padding=1)
        
        self.enc1 = MedNeXtBlock(base_channels)
        self.down1 = nn.Conv3d(base_channels, base_channels * 2, kernel_size=2, stride=2)

        self.enc2 = MedNeXtBlock(base_channels * 2)
        self.down2 = nn.Conv3d(base_channels * 2, base_channels * 4, kernel_size=2, stride=2)

        self.enc3 = MedNeXtBlock(base_channels * 4)
        self.down3 = nn.Conv3d(base_channels * 4, base_channels * 8, kernel_size=2, stride=2)

        self.enc4 = MedNeXtBlock(base_channels * 8)
        self.down4 = nn.Conv3d(base_channels * 8, base_channels * 16, kernel_size=2, stride=2)

        self.bottleneck = nn.Sequential(
            MedNeXtBlock(base_channels * 16),
            AxialAttention3D(base_channels * 16) if use_attention else nn.Identity()
        )

        self.up4 = nn.ConvTranspose3d(base_channels * 16, base_channels * 8, kernel_size=2, stride=2)
        self.dec4 = MedNeXtBlock(base_channels * 8)

        self.up3 = nn.ConvTranspose3d(base_channels * 8, base_channels * 4, kernel_size=2, stride=2)
        self.dec3 = MedNeXtBlock(base_channels * 4)

        self.up2 = nn.ConvTranspose3d(base_channels * 4, base_channels * 2, kernel_size=2, stride=2)
        self.dec2 = MedNeXtBlock(base_channels * 2)

        self.up1 = nn.ConvTranspose3d(base_channels * 2, base_channels, kernel_size=2, stride=2)
        self.dec1 = MedNeXtBlock(base_channels)

        self.final = nn.Conv3d(base_channels, out_channels, kernel_size=1)

    def forward(self, x):
        x0 = self.stem(x)       
        x1 = self.enc1(x0)
        x2 = self.enc2(self.down1(x1))
        x3 = self.enc3(self.down2(x2))
        x4 = self.enc4(self.down3(x3))
        x5 = self.bottleneck(self.down4(x4))

        u4 = self._pad_or_crop(self.up4(x5), x4)
        d4 = self.dec4(u4 + x4)

        u3 = self._pad_or_crop(self.up3(d4), x3)
        d3 = self.dec3(u3 + x3)

        u2 = self._pad_or_crop(self.up2(d3), x2)
        d2 = self.dec2(u2 + x2)

        u1 = self._pad_or_crop(self.up1(d2), x1)
        d1 = self.dec1(u1 + x1)

        out = self.final(d1)
        return out

    def _pad_or_crop(self, src, target):
        diff = [s - t for s, t in zip(src.shape[2:], target.shape[2:])]
        for i in range(3):
            if diff[i] > 0:
                src = src.narrow(2 + i, diff[i] // 2, target.shape[2 + i])
            elif diff[i] < 0:
                pad = [0, 0, 0, 0, 0, 0]
                pad[-(2 * i + 1)] = -diff[i] // 2
                pad[-(2 * i + 2)] = -diff[i] + (-diff[i] // 2)
                src = nn.functional.pad(src, pad)
        return src

model = MedNeXtWithAttentionUNet(
    in_channels=2,
    out_channels=4,
    base_channels=32,
    use_attention=True
).to(device)

dummy = torch.randn(1, 2, 128, 128, 128).to(device)
out = model(dummy)
print("Output shape:", out.shape)

In [ ]:
class BratsModelTrainer:
    def __init__(self, model, train_loader, val_loader, num_classes=4, device=None, parameters=parameters):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.num_classes = num_classes
        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.parameters = parameters
        self.dice_loss = DiceLoss(to_onehot_y=True, softmax=True)
        self.ce_loss = nn.CrossEntropyLoss()
        
        self.dice_metric = DiceMetric(
            include_background=False,
            reduction="none"
        )
        self.hd95_metric = HausdorffDistanceMetric(
            include_background=False,
            percentile=95,
            reduction="none",
            get_not_nans=False
        )

        self.reset_metrics()
        
    def combined_loss(self, pred, target):
            target_ce = target.squeeze(1).long()
            return 0.5 * self.dice_loss(pred, target) + 0.5 * self.ce_loss(pred, target_ce)
            
    def reset_metrics(self):
        self.metrics = {
            'train_loss': [],
            'val_loss': [],
            'dice': {f'class_{i}': [] for i in range(1, self.num_classes)},
            'hd95': {f'class_{i}': [] for i in range(1, self.num_classes)},
            'mean_dice': [],
            'mean_hd95': []
        }

    def train(self):
        if self.parameters['optimizer'] == 'adam':
            optimizer = torch.optim.Adam(self.model.parameters(), lr=self.parameters['lr'])
        elif self.parameters['optimizer'] == 'adamw':
            optimizer = torch.optim.AdamW(self.model.parameters(), lr=self.parameters['lr'])
        else:
            print('optimizer options can only be adam and adamw')
            return

        if self.parameters['scheduler'] == 'reduceLROnPlateau':
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                    optimizer, 
                    mode='min', 
                    factor=0.1, 
                    patience=5, 
                    threshold=1e-4
                )
        elif self.parameters['scheduler'] == 'cosineAnnealing':
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                    optimizer, 
                    T_max=self.parameters['num_epochs'], 
                    eta_min=1e-5
                )
        else:
            raise ValueError("Scheduler must be 'reduceLROnPlateau' or 'cosineAnnealing'")
    
        best_val_loss = 0
        epochs_no_improve = 0
        patience = self.parameters['stopping_patience']
    
        for epoch in tqdm(range(self.parameters['num_epochs']), desc="Training"):
            self.model.train()
            epoch_train_loss = 0.0
    
            pbar = tqdm(self.train_loader, desc=f"Epoch {epoch+1}", leave=False)
            for batch in pbar:
                inputs = batch["vol"][tio.DATA].to(self.device).float()
                labels = batch["label"][tio.DATA].long().to(self.device)
                labels = torch.where(labels == 4, torch.tensor(3, device=labels.device), labels)
    
                optimizer.zero_grad()
                outputs = self.model(inputs)
    
                loss = self.combined_loss(outputs, labels)
                loss.backward()
                optimizer.step()
    
                epoch_train_loss += loss.item()
                pbar.set_postfix(loss=loss.item(), lr=optimizer.param_groups[0]['lr'])
    
            avg_train_loss = epoch_train_loss / len(self.train_loader)
            self.metrics['train_loss'].append(avg_train_loss)
    
            val_loss, mean_dice, mean_hd95 = self.validate()
            self.metrics['val_loss'].append(val_loss)
    
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                epochs_no_improve = 0
                torch.save(self.model.state_dict(), f"{self.parameters['out_dir']}/best_model.pth")
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= patience:
                    print(f"\nEarly stopping at epoch {epoch+1}")
                    break
    
            if self.parameters['scheduler'] == 'reduceLROnPlateau':
                scheduler.step(val_loss)
            else:
                scheduler.step()

            current_lr = optimizer.param_groups[0]['lr']

            tqdm.write(
                f"Training Loss: {(epoch_train_loss / len(self.train_loader)):.4f} | "
                f"Validation Loss : {val_loss:.4f} | "
                f"Mean HD95 : {mean_hd95:.4f} | "
                f"Mean Dice: {mean_dice:.4f} | "
                f"Learning rate : {current_lr:.6f}"
              )

        # return self.metrics

    def validate(self):
        self.model.eval()
        epoch_val_loss = 0.0
    
        self.dice_metric.reset()
    
        dice_class_values = []
        hd95_class_values = []
    
        with torch.no_grad():
            for batch in self.val_loader:
                inputs = batch["vol"][tio.DATA].to(self.device).float()
                labels = batch["label"][tio.DATA].long().to(self.device)
                labels = torch.where(labels == 4, torch.tensor(3, device=labels.device), labels)
    
                outputs = self.model(inputs)
                loss = self.combined_loss(outputs, labels)
                epoch_val_loss += loss.item()
    
                outputs_softmax = torch.softmax(outputs, dim=1)
    
                one_hot_labels = torch.nn.functional.one_hot(
                    labels.squeeze(1), num_classes=self.num_classes
                ).permute(0, 4, 1, 2, 3).float()
    
                self.dice_metric(outputs_softmax, one_hot_labels)
    
                preds = outputs_softmax.argmax(dim=1)  
    
                for class_idx in range(1, self.num_classes):  
                    pred_bin = (preds == class_idx).float().unsqueeze(1)  
                    label_bin = (labels == class_idx).float()            \
                    
                    hd = compute_hausdorff_distance(
                        pred_bin, label_bin, percentile=95.0, include_background=False
                    ) 
                    hd95_class_values.append(hd)
    
        val_loss = epoch_val_loss / len(self.val_loader)
    
        dice_values_raw = self.dice_metric.aggregate()
        dice_values = torch.nan_to_num(dice_values_raw, nan=0.0) 
    
        if hd95_class_values:
            hd95_values_stack = torch.stack(hd95_class_values)  
            hd95_values = hd95_values_stack.squeeze(-1).squeeze(-1).squeeze(-1).permute(1, 0)  
            hd95_values = torch.nan_to_num(hd95_values, nan=0.0)
        else:
            hd95_values = torch.zeros((1, self.num_classes - 1), device=self.device)
    
        num_reported_classes = dice_values.shape[1]
        for class_idx in range(num_reported_classes):
            dice_score = dice_values[:, class_idx].mean().item()
            hd95_score = hd95_values[:, class_idx].mean().item()
    
            self.metrics['dice'][f'class_{class_idx + 1}'].append(dice_score)
            self.metrics['hd95'][f'class_{class_idx + 1}'].append(hd95_score)
    
        mean_dice = dice_values.mean().item()
        mean_hd95 = hd95_values.mean().item()
    
        self.metrics['mean_dice'].append(mean_dice)
        self.metrics['mean_hd95'].append(mean_hd95)
    
        return val_loss, mean_dice, mean_hd95


    @staticmethod
    def predict(model, input_volume, device=None, parameters=None, slice_idx=None):
        device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model.to(device)
        model.eval()

        with torch.no_grad():
            if len(input_volume.shape) == 4:
                input_volume = input_volume.unsqueeze(0)
            input_volume = input_volume.to(device).float()
            outputs = model(input_volume)
            preds = torch.argmax(outputs, dim=1)

        preds = preds.squeeze(0).cpu().numpy()

        slice_idx = slice_idx if slice_idx is not None else input_volume.shape[-3] // 2
        input_slice = input_volume[0, :, :, slice_idx].cpu().numpy()
        pred_slice = preds[:, :, slice_idx]

        BratsModelTrainer.visualize_prediction(
            input_slice=input_slice,
            ground_truth=np.zeros_like(pred_slice),
            prediction=pred_slice,
            parameters=parameters
        )

    def plot_and_save_loss(self):
        save_path = f"{self.parameters['out_dir']}/losses.png"
        plt.figure(figsize=(10, 6))

        plt.plot(self.metrics['train_loss'], label='Train Loss')
        plt.plot(self.metrics['val_loss'], label='Validation Loss')
        plt.title('Training and Validation Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()
        plt.tight_layout()
        plt.savefig(save_path)
        # plt.close()
        plt.show()

    def save_metrics_and_params(self):
        metrics_path = f"{self.parameters['out_dir']}/metrics.csv"
        params_path = f"{self.parameters['out_dir']}/model_parameters.csv"

        metrics_dict = {
            'train_loss': self.metrics['train_loss'],
            'val_loss': self.metrics['val_loss'],
            'mean_dice': self.metrics['mean_dice'],
            'mean_hd95': self.metrics['mean_hd95'],
        }

        for class_idx in range(1, self.num_classes):
            metrics_dict[f'dice_class_{class_idx}'] = self.metrics['dice'][f'class_{class_idx}']
            metrics_dict[f'hd95_class_{class_idx}'] = self.metrics['hd95'][f'class_{class_idx}']

        metrics_df = pd.DataFrame(metrics_dict)
        metrics_df.to_csv(metrics_path, index=False)

        params_df = pd.DataFrame(self.parameters, index=[0])
        params_df.to_csv(params_path, index=False)
        
    def generate_predictions_and_save(self, test_loader):
        self.model.eval()
        output_dir = os.path.join(self.parameters['out_dir'], "predictions")
        os.makedirs(output_dir, exist_ok=True)

        with torch.no_grad():
            for batch in tqdm(test_loader, desc="Making Predictions"):
                inputs = batch["vol"][tio.DATA].to(self.device).float()
                patient_ids = batch["patient_id"] 
                affines = batch["vol"]["affine"]

                outputs = self.model(inputs)
                preds = torch.argmax(outputs, dim=1).cpu().numpy()

                for i in range(inputs.shape[0]):
                    pred_np = preds[i].astype(np.uint8)
                    affine = affines[i].numpy() if hasattr(affines[i], 'numpy') else affines[i]

                    filename = f"{patient_ids[i]}.nii.gz"
                    out_path = os.path.join(output_dir, filename)

                    nib.save(nib.Nifti1Image(pred_np, affine), out_path)


    @staticmethod
    def visualize_prediction(input_slice, prediction, alpha=0.5, parameters=parameters, pred_id=None):
        colors = ['black', 'red', 'green', 'blue', 'yellow']
        # cmap = ListedColormap(colors[:len(np.unique(ground_truth))])
        if pred_id:
            save_path = f"{parameters['out_dir']}/prediction_{pred_id}.png" if parameters else "prediction.png"
        else:
            save_path = f"{parameters['out_dir']}/prediction.png" if parameters else "prediction.png"
            
        plt.figure(figsize=(18, 6))

        plt.subplot(1, 2, 1)
        plt.imshow(input_slice, cmap='gray')
        plt.title('Input Image')
        plt.axis('off')

        plt.subplot(1, 2, 3)
        plt.imshow(prediction, cmap='gray')
        plt.title('Prediction')
        plt.axis('off')

        plt.tight_layout()
        plt.savefig(save_path)
        # plt.close()


In [ ]:
brats_model = BratsModelTrainer(model=model, train_loader=train_loader, val_loader=val_loader)

brats_model.train()

In [ ]:
brats_model.plot_and_save_loss()

In [ ]:
brats_model.save_metrics_and_params()

In [ ]:
brats_model.generate_predictions_and_save(test_loader)

In [ ]:
img_number = 1
pred = os.listdir(os.path.join(parameters['out_dir'], 'predictions'))[img_number]
full_pred_path = os.path.join(parameters['out_dir'], 'predictions', pred)
pred_data = nib.load(full_pred_path).get_fdata()

test_loader = SubjectsLoader(test_dataset, batch_size=1, shuffle=False)

input_data = []
for batch in iter(test_loader):
    input_img_path = batch['patient_id'][0]
    if input_img_path == pred[:-7]:
        input_data = batch['vol'][tio.DATA]
        break

pred_slice = pred_data[:, :, pred_data.shape[2] // 2]
input_slice = input_data[0, 0, :, :, input_data.shape[4] // 2]

brats_model.visualize_prediction(input_slice=input_slice, prediction=pred_slice, parameters=parameters)

In [ ]:
timestamp_folder = os.listdir(os.path.join(parameters['out_dir'], '../'))[0]
!zip -r ./results.zip "./output/{timestamp_folder}/"
# print(timestamp_folder)

In [ ]:
metrics = pd.read_csv('/kaggle/working/output/2025-07-03_08-26/metrics.csv')
metrics

In [ ]:
model_params = pd.read_csv('/kaggle/working/output/2025-07-03_08-26/model_parameters.csv')
model_params